In [0]:
import faker
from rand_engine.core.distinct_core import DistinctCore
from rand_engine.core.numeric_core import NumericCore
from rand_engine.core.datetime_core import DatetimeCore
from rand_engine.core.distinct_utils import DistinctUtils
from rand_engine.main.data_generator import DataGenerator
from pandas import DataFrame as PandasDF
import numpy as np
import pandas as pd
from datetime import datetime as dt, timedelta


class FakeOrders:

    def __init__(self):
        self.faker = faker.Faker(locale="pt_BR")

    def metadata(self, start_date, end_date):
        return {
            "order_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=16)
            },
            "user_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=10)
            },
            "product_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=3)
            },
            "user_type": {     
            "method": DistinctCore.gen_distincts_untyped,
            "parms": dict(distinct=DistinctUtils.handle_distincts_lvl_1({"standard": 80,"premium":15, "gold": 5, None: 7}, 1))
            },
            "device": {
                "method": DistinctCore.gen_distincts_typed,
                "parms": dict(distinct=["IOS", "Android", "Desktop"])
            },
            "traffic_source": {
                "method": DistinctCore.gen_distincts_typed,
                "parms": dict(distinct=["website", "linkedin", "email"])
            },
            "created_at_2": dict(
                method=DatetimeCore.gen_timestamps,
                parms=dict(start=start_date, end=end_date, format="%Y-%m-%d")
            )
        }

    def transformer(self, **kwargs):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            start_time = (dt.now() - timedelta(minutes=10))
            end_time = dt.now()
            random_times = pd.to_datetime(np.random.uniform(start_time.timestamp(), end_time.timestamp(), size=len(df)), unit='s').strftime('%Y-%m-%dT%H:%M:%S')
            df["created_at"] = random_times
            return df
        return wrapped_transformer
    

from datetime import datetime as dt
from uuid import uuid4


def widgets():
    dbutils.widgets.text("odate", "2025-09-23")
    dbutils.widgets.text("catalog", "prd")
    dbutils.widgets.text("schema", "stage")
    dbutils.widgets.text("volume", "raw_data_managed")
    dbutils.widgets.text("vol_path","raw_data/retail/orders_csv/orders")

def main():

    ODATE = dbutils.widgets.get("odate")
    CATALOG = dbutils.widgets.get("catalog")
    SCHEMA = dbutils.widgets.get("schema")
    VOLUME = dbutils.widgets.get("volume")
    VOL_PATH = dbutils.widgets.get("vol_path")
    SIZE = 10**5

    BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{VOL_PATH}"
    print(f"Base Path: {BASE_PATH}")
    print(f"Order Data: {ODATE}")
    file_path = f"{BASE_PATH}/orders_{uuid4()}.csv"
    end_date = ODATE
    start_date = (dt.strptime(end_date, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")

    orders_spec = FakeOrders()
    df_orders = (
        DataGenerator(orders_spec.metadata(start_date, end_date))
            .generate_pandas_df(size=SIZE, transformer=orders_spec.transformer())
            .get_df()
            .to_csv(file_path, index=False, header=True, mode="w")
    )
    print(f"File Path: {file_path} written. Size: {SIZE} records.")

widgets()
main()
